Setup ambiente

In [2]:
!pip -q install -U transformers datasets evaluate accelerate scikit-learn

import os, json, numpy as np
import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.6 MB/s eta 0:00:00


Path + LOAD DATASET (HF datasets) + LABEL MAP

In [6]:
import os
from datasets import load_dataset, DatasetDict

# =========================
# PATH (tuoi file già creati)
# =========================
BASE = "/content/Semantic"
OUT_DIR = os.path.join(BASE, "dataset_002_multi")

train_clf_path = os.path.join(OUT_DIR, "train_classification.json")
test_clf_path  = os.path.join(OUT_DIR, "test_classification.json")

assert os.path.exists(train_clf_path), f"Non trovo: {train_clf_path}"
assert os.path.exists(test_clf_path),  f"Non trovo: {test_clf_path}"

# =========================
# LABEL NORMALIZATION (4 -> 2)
# =========================
LABEL_CANON = {
    "inclusiva": "inclusive",
    "inclusive": "inclusive",
    "non_inclusiva": "not_inclusive",
    "not_inclusive": "not_inclusive",
}

def canonize_label(lab: str) -> str:
    lab = str(lab).strip().lower()
    if lab not in LABEL_CANON:
        raise ValueError(f"Label non riconosciuta: {lab}")
    return LABEL_CANON[lab]

def canonize_example(ex):
    ex["text"] = str(ex["text"]).strip()
    ex["label"] = canonize_label(ex["label"])
    return ex

# =========================
# LOAD TRAIN/TEST
# =========================
ds = load_dataset("json", data_files={"train": train_clf_path, "test": test_clf_path})

# Canonizza SEMPRE (anche se i file su disco sono "sporchi")
ds = ds.map(canonize_example)
ds = ds.filter(lambda ex: ex["text"] != "")

# =========================
# VALIDATION (10% dal train)
# =========================
split = ds["train"].train_test_split(test_size=0.1, seed=42, shuffle=True)
ds = DatasetDict({"train": split["train"], "validation": split["test"], "test": ds["test"]})

# =========================
# LABEL2ID fisso (2 classi)
# =========================
LABEL2ID = {"not_inclusive": 0, "inclusive": 1}
ID2LABEL = {0: "not_inclusive", 1: "inclusive"}

def encode_label(ex):
    ex["label_id"] = LABEL2ID[ex["label"]]
    return ex

ds = ds.map(encode_label)

# =========================
# DEBUG
# =========================
print(ds)
print("Label uniche (train):", sorted(set(ds["train"]["label"])))
print("LABEL2ID:", LABEL2ID)
print("Esempio:", ds["train"][0])


Map:   0%|          | 0/34572 [00:00<?, ? examples/s]

Map:   0%|          | 0/8644 [00:00<?, ? examples/s]

Filter:   0%|          | 0/34572 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8644 [00:00<?, ? examples/s]

Map:   0%|          | 0/31114 [00:00<?, ? examples/s]

Map:   0%|          | 0/3458 [00:00<?, ? examples/s]

Map:   0%|          | 0/8644 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_id'],
        num_rows: 31114
    })
    validation: Dataset({
        features: ['text', 'label', 'label_id'],
        num_rows: 3458
    })
    test: Dataset({
        features: ['text', 'label', 'label_id'],
        num_rows: 8644
    })
})
Label uniche (train): ['inclusive', 'not_inclusive']
LABEL2ID: {'not_inclusive': 0, 'inclusive': 1}
Esempio: {'text': "L'ordine in casa può essere gestito da tutti.", 'label': 'inclusive', 'label_id': 1}


Check leakage

In [7]:
train_texts = set(ds["train"]["text"])
val_texts   = set(ds["validation"]["text"])
test_texts  = set(ds["test"]["text"])

print("Overlap train∩val :", len(train_texts & val_texts))
print("Overlap train∩test:", len(train_texts & test_texts))
print("Overlap val∩test  :", len(val_texts & test_texts))


Overlap train∩val : 73
Overlap train∩test: 164
Overlap val∩test  : 22


TOKENIZE

In [8]:
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL_NAME = "xlm-roberta-base"   # ✅ multilingua, molto solido
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True)

tokenized = ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=[c for c in ds["train"].column_names if c not in ["label_id"]]
)
tokenized = tokenized.rename_column("label_id", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/31114 [00:00<?, ? examples/s]

Map:   0%|          | 0/3458 [00:00<?, ? examples/s]

Map:   0%|          | 0/8644 [00:00<?, ? examples/s]

TRAIN + METRICHE

In [10]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

args = TrainingArguments(
    output_dir="./clf_out",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="steps",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-135059562.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.068300,0.058958,0.988143,0.988201,0.988143,0.988142
2,0.019400,0.041425,0.993638,0.993639,0.993638,0.993638
3,0.000100,0.044443,0.993638,0.993640,0.993638,0.993638


TrainOutput(global_step=5835, training_loss=0.04729310039713366, metrics={'train_runtime': 1082.0801, 'train_samples_per_second': 86.262, 'train_steps_per_second': 5.392, 'total_flos': 1376387375021280.0, 'train_loss': 0.04729310039713366, 'epoch': 3.0})

Evaluate + Save + Pipeline test

In [12]:
# --- evaluate sul test ---
test_metrics = trainer.evaluate(tokenized["test"])
print("TEST METRICS:", test_metrics)

# report dettagliato
pred = trainer.predict(tokenized["test"])
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=-1)
print(classification_report(y_true, y_pred, target_names=[ID2LABEL[i] for i in range(len(ID2LABEL))]))

# --- save (modello + tokenizer + label map) ---
SAVE_DIR = "./model_classifier_multi"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(os.path.join(SAVE_DIR, "label_map.json"), "w") as f:
    json.dump(ID2LABEL, f, indent=2)

print("Salvato in:", SAVE_DIR)

# --- pipeline test (uso reale) ---
from transformers import pipeline

clf = pipeline("text-classification", model=SAVE_DIR, tokenizer=SAVE_DIR, device=0 if torch.cuda.is_available() else -1)

samples = [
    "Blonde women are usually not very smart.",
    "Hair color does not determine a person's intelligence.",
    "Le donne bionde di solito non sono molto intelligenti.",
    "Il colore dei capelli non determina l'intelligenza di una persona."
]

for s in samples:
    print(s, "->", clf(s))


TEST METRICS: {'eval_loss': 0.03631875291466713, 'eval_accuracy': 0.9931744562702453, 'eval_precision': 0.9931820864647918, 'eval_recall': 0.9931744562702453, 'eval_f1': 0.9931744298700427, 'eval_runtime': 8.5717, 'eval_samples_per_second': 1008.439, 'eval_steps_per_second': 31.616, 'epoch': 3.0}
               precision    recall  f1-score   support

not_inclusive       1.00      0.99      0.99      4322
    inclusive       0.99      1.00      0.99      4322

     accuracy                           0.99      8644
    macro avg       0.99      0.99      0.99      8644
 weighted avg       0.99      0.99      0.99      8644

Salvato in: ./model_classifier_multi


The tokenizer you are loading from './model_classifier_multi' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Device set to use cuda:0


Blonde women are usually not very smart. -> [{'label': 'not_inclusive', 'score': 0.9999090433120728}]
Hair color does not determine a person's intelligence. -> [{'label': 'inclusive', 'score': 0.999982476234436}]
Le donne bionde di solito non sono molto intelligenti. -> [{'label': 'not_inclusive', 'score': 0.9999017715454102}]
Il colore dei capelli non determina l'intelligenza di una persona. -> [{'label': 'inclusive', 'score': 0.9999831914901733}]
